# Phase04

## 1. Setup

In [25]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
import geopandas as gpd
import statsmodels.formula.api as smf

DATA_ROOT = Path("../data")
GEO_DIR   = DATA_ROOT / "geometries"

REC_OD    = DATA_ROOT / "recent_2025-03" / "Viajes_distritos_2025-03_v2.parquet"   # March 2025 OD
CAT_FILE  = DATA_ROOT / "poi" / "fsq_places_spain_categories.parquet"              # Spain POIs + category

## 2. POIs per district by type

In [26]:
# top-level category = text before the first ">" in the label
poi = pd.read_parquet(CAT_FILE)
poi["category_top"] = poi["category_label"].str.split(">").str[0].str.strip()
poi = poi.dropna(subset=["category_top"])
poi = poi[poi["category_top"] != ""]
print("Spain POIs with a category:", f"{len(poi):,}")
print(poi["category_top"].value_counts())

Spain POIs with a category: 1,814,306
category_top
Dining and Drinking                   467003
Retail                                379426
Business and Professional Services    373342
Travel and Transportation             149200
Community and Government              141223
Landmarks and Outdoors                132987
Health and Medicine                    68077
Arts and Entertainment                 57494
Sports and Recreation                  39391
Event                                   6163
Name: count, dtype: int64


In [27]:
# assign each POI to the district polygon that contains it (point-in-polygon, metres CRS)
gdf = gpd.read_file(GEO_DIR / "distritos_2.geojson")[["id", "geometry"]].to_crs(25830)
poi_gdf = gpd.GeoDataFrame(poi, geometry=gpd.points_from_xy(poi["longitude"], poi["latitude"]),
                           crs=4326).to_crs(25830)
joined = gpd.sjoin(poi_gdf, gdf, predicate="within", how="inner")

# wide count matrix: one row per district, one column per top-level category
counts = (joined.groupby(["id", "category_top"]).size()
                 .unstack(fill_value=0)
                 .reindex(gdf["id"]).fillna(0).astype(int))

# safe column names for the regression formula:  "Dining and Drinking" -> "cat_dining_and_drinking"
def safe(name):
    return "cat_" + re.sub(r"[^0-9a-z]+", "_", name.lower()).strip("_")

cat_cols = {c: safe(c) for c in counts.columns}
district_cat = counts.rename(columns=cat_cols)
CAT = list(cat_cols.values())                     # the category column names, in a fixed order
district_cat["poi_total"] = district_cat[CAT].sum(axis=1)
district_cat.index.name = "district"
print(f"{len(CAT)} categories, {len(district_cat):,} districts")
print(district_cat[CAT].sum().sort_values(ascending=False))

10 categories, 3,792 districts
category_top
cat_dining_and_drinking                   464782
cat_retail                                378990
cat_business_and_professional_services    372970
cat_travel_and_transportation             147210
cat_community_and_government              140973
cat_landmarks_and_outdoors                130480
cat_health_and_medicine                    68026
cat_arts_and_entertainment                 57136
cat_sports_and_recreation                  39203
cat_event                                   6066
dtype: int64


## 3. Geometry and population

In [28]:
EARTH_KM = 6371.0088

def haversine_km(lon1, lat1, lon2, lat2):
    """Great-circle distance (km) — valid across UTM zones and out to the islands."""
    lon1, lat1, lon2, lat2 = map(np.radians, (lon1, lat1, lon2, lat2))
    a = np.sin((lat2 - lat1) / 2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin((lon2 - lon1) / 2)**2
    return 2 * EARTH_KM * np.arcsin(np.sqrt(a))

# fixed centroid per district in lon/lat (for geodesic distance)
cent = gdf.set_index("id").geometry.centroid.to_crs(4326)
dist_lon, dist_lat = cent.x, cent.y

# population per district
district_pop = pd.read_csv(GEO_DIR / "poblacion_distritos.csv", sep="|", header=None,
                           names=["district", "population"], dtype={"district": str})
district_pop = pd.to_numeric(district_pop.set_index("district")["population"], errors="coerce")
print(f"districts with population: {district_pop.notna().sum():,}")

districts with population: 3,743


## 4. Trips → district pairs (March 2025)

In [29]:
def od_pairs_district(parquet_path):
    """Month of district OD -> per-day flows between distinct districts."""
    od = pd.read_parquet(parquet_path, columns=["date", "id_origin", "id_destination", "n_trips"])
    n_days = od["date"].nunique()
    for c in ["id_origin", "id_destination"]:
        od[c] = od[c].astype("category")               # 109M rows -> keep the groupby light
    pairs = od.groupby(["id_origin", "id_destination"], observed=True, as_index=False)["n_trips"].sum()
    pairs["id_origin"] = pairs["id_origin"].astype(str)
    pairs["id_destination"] = pairs["id_destination"].astype(str)
    pairs = pairs[pairs["id_origin"] != pairs["id_destination"]]   # drop within-district
    pairs["trips_per_day"] = pairs["n_trips"] / n_days
    return pairs

rec_pairs = od_pairs_district(REC_OD)
print(f"March 2025: {len(rec_pairs):,} district pairs")

March 2025: 3,359,772 district pairs


## 5. Model frame

In [30]:
def build_frame(pairs):
    """District pairs + centroid distance + origin population + destination masses, logged.
    Every model below fits on this SAME row set, so R2 / AIC / gamma compare directly.

    Category counts use log1p (log(1+count)) because many districts have zero POIs of a
    given type; population and distance must be strictly positive (plain log)."""
    df = pairs.copy()
    df["dist_km"] = haversine_km(df["id_origin"].map(dist_lon), df["id_origin"].map(dist_lat),
                                 df["id_destination"].map(dist_lon), df["id_destination"].map(dist_lat))
    df["pop_i"] = df["id_origin"].map(district_pop)
    df["pop_j"] = df["id_destination"].map(district_pop)
    df["poi_total_j"] = df["id_destination"].map(district_cat["poi_total"])

    # keep only rows where the plain-log terms are defined
    df = df.dropna(subset=["dist_km", "pop_i", "pop_j", "poi_total_j"])
    df = df[(df["trips_per_day"] > 0) & (df["dist_km"] > 0) & (df["pop_i"] > 0) & (df["pop_j"] > 0)]

    df["log_trips"] = np.log(df["trips_per_day"])
    df["log_dij"]   = np.log(df["dist_km"])
    df["log_pop_i"] = np.log(df["pop_i"])
    df["log_pop_j"] = np.log(df["pop_j"])
    df["log_poi_total_j"] = np.log1p(df["poi_total_j"])

    # squared log-distance: lets the decay steepen at short range and flatten at long range
    df["log_dij2"] = df["log_dij"] ** 2

    # one log-count column per category, evaluated at the DESTINATION j
    for c in CAT:
        df["log_" + c] = np.log1p(df["id_destination"].map(district_cat[c]).fillna(0))
    return df

rec_model = build_frame(rec_pairs)
print(f"model rows (n): {len(rec_model):,}")

model rows (n): 3,126,370


## 6. Gravity models

In [31]:
# origin is always population (log_pop_i); the models differ in destination mass and distance form.
cat_terms = " + ".join("log_" + c for c in CAT)

m_pop  = smf.ols("log_trips ~ log_pop_i + log_pop_j + log_dij",            data=rec_model).fit()  # population
m_tot  = smf.ols("log_trips ~ log_pop_i + log_poi_total_j + log_dij",      data=rec_model).fit()  # POIs, total count
m_cat  = smf.ols(f"log_trips ~ log_pop_i + {cat_terms} + log_dij",         data=rec_model).fit()  # POIs, by type
m_d2   = smf.ols("log_trips ~ log_pop_i + log_pop_j + log_dij + log_dij2", data=rec_model).fit()  # population + dist^2

# full statsmodels regression tables for every model
for label, m in [("pop_i + pop_j", m_pop),
                 ("pop_i + POI_j (total count)", m_tot),
                 ("pop_i + POI_j (by types)", m_cat),
                 ("pop_i + pop_j + dist^2", m_d2)]:
    print("=" * 78, f"\n{label}\n", "=" * 78, sep="")
    print(m.summary())
    print()

pop_i + pop_j
                            OLS Regression Results                            
Dep. Variable:              log_trips   R-squared:                       0.661
Model:                            OLS   Adj. R-squared:                  0.661
Method:                 Least Squares   F-statistic:                 2.034e+06
Date:                Thu, 02 Jul 2026   Prob (F-statistic):               0.00
Time:                        11:45:33   Log-Likelihood:            -4.8534e+06
No. Observations:             3126370   AIC:                         9.707e+06
Df Residuals:                 3126366   BIC:                         9.707e+06
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept     -2.3868      0.009   -27

In [32]:
def fit_row(m, label):
    return {"model": label, "n": int(m.nobs), "params": len(m.params) - 1,
            "R2": m.rsquared, "adj_R2": m.rsquared_adj, "AIC": m.aic}

summary = pd.DataFrame([fit_row(m_pop, "pop_i + pop_j"),
                        fit_row(m_tot, "pop_i + POI_j (total count)"),
                        fit_row(m_cat, "pop_i + POI_j (by types)"),
                        fit_row(m_d2,  "pop_i + pop_j + dist^2")])
summary.set_index("model").round({"R2": 4, "adj_R2": 4, "AIC": 1})

,n,params,R2,adj_R2,AIC
model,,,,,
pop_i + pop_j,3126370,3,0.6613,0.6613,9706804.6
pop_i + POI_j (total count),3126370,3,0.6646,0.6646,9675664.1
pop_i + POI_j (by types),3126370,12,0.6685,0.6685,9639569.5
pop_i + pop_j + dist^2,3126370,4,0.7270,0.7270,9032568.3


In [33]:
# per-category attraction elasticities from the by-type model (sorted by size)
cat_effects = (pd.DataFrame({"beta": m_cat.params, "p_value": m_cat.pvalues})
                 .loc[["log_" + c for c in CAT]]
                 .assign(category=[c.replace("cat_", "") for c in CAT])
                 .set_index("category").sort_values("beta", ascending=False))
cat_effects.round({"beta": 3, "p_value": 4})

,beta,p_value
category,,
travel_and_transportation,0.238,0.0000
sports_and_recreation,0.095,0.0000
retail,0.083,0.0000
community_and_government,0.062,0.0000
health_and_medicine,0.054,0.0000
business_and_professional_services,0.045,0.0000
landmarks_and_outdoors,0.012,0.0000
event,-0.004,0.0001
dining_and_drinking,-0.022,0.0000


In [34]:
# category counts are correlated across districts (big places have lots of everything);
# VIF flags coefficients whose individual size is unreliable because of that overlap.
from statsmodels.stats.outliers_influence import variance_inflation_factor
X = rec_model[["log_" + c for c in CAT]].to_numpy()
vif = pd.Series([variance_inflation_factor(X, k) for k in range(X.shape[1])],
                index=[c.replace("cat_", "") for c in CAT], name="VIF")
vif.sort_values(ascending=False).round(1)

retail                                226.6
business_and_professional_services    184.3
dining_and_drinking                   157.1
community_and_government              113.7
landmarks_and_outdoors                 60.3
travel_and_transportation              48.0
arts_and_entertainment                 30.2
health_and_medicine                    26.7
sports_and_recreation                  20.9
event                                   3.3
Name: VIF, dtype: float64

## 7. Gravity by age × income (4×3 grid)

Same gravity model `log_trips ~ log_pop_i + log_pop_j + log_dij`, fit separately within each **age band × income band** cell (masses = total district population, identical across cells). The result is a summary table of the coefficients per cell, plus γ (distance decay) and R² as age×income grids.

In [35]:
# same March 2025 month, re-parsed to keep the traveler's age (edad) and residence-area income
# band (renta).  edad: 0-25/25-45/45-65/65-100/NA ; renta = net income EUR000/person/yr: <10/10-15/>15.
# renta is a RESIDENCE (origin) attribute; NA age rows (~29% of trips, suppressed) are excluded below.
# NOTE: this file keeps the raw MITMA Spanish column names (fecha/origen/destino) -> rename on read.
AGE_INC = DATA_ROOT / "recent_2025-03" / "Viajes_distritos_2025-03_age_income_v2.parquet"

seg = pd.read_parquet(AGE_INC, columns=["fecha", "origen", "destino", "edad", "renta", "n_trips"])
seg = seg.rename(columns={"origen": "id_origin", "destino": "id_destination"})
seg_days = seg["fecha"].nunique()
seg = seg.drop(columns="fecha")
for c in ["id_origin", "id_destination", "edad", "renta"]:
    seg[c] = seg[c].astype("category")
# collapse the daily rows -> one flow per (origin, destination, age, income) cell
seg = seg.groupby(["id_origin", "id_destination", "edad", "renta"], observed=True,
                  as_index=False)["n_trips"].sum()
print(f"{len(seg):,} origin x destination x age x income rows over {seg_days} days")
print("age bands :", sorted(seg["edad"].unique()))
print("income    :", sorted(seg["renta"].unique()))

10,815,132 origin x destination x age x income rows over 31 days
age bands : ['0-25', '25-45', '45-65', '65-100', 'NA']
income    : ['10-15', '<10', '>15']


In [36]:
# Fit the SAME gravity model  log_trips ~ log_pop_i + log_pop_j + log_dij  separately in each
# age x income cell.  Masses are the TOTAL district populations (identical across cells), so the
# only thing changing cell-to-cell is how each demographic's own flows respond to mass and distance.
def fit_cell(sub):
    d = sub.copy()
    d["id_origin"] = d["id_origin"].astype(str)
    d["id_destination"] = d["id_destination"].astype(str)
    d = d[d["id_origin"] != d["id_destination"]]                       # drop within-district
    d["trips_per_day"] = d["n_trips"] / seg_days
    d["dist_km"] = haversine_km(d["id_origin"].map(dist_lon), d["id_origin"].map(dist_lat),
                                d["id_destination"].map(dist_lon), d["id_destination"].map(dist_lat))
    d["pop_i"] = d["id_origin"].map(district_pop)
    d["pop_j"] = d["id_destination"].map(district_pop)
    d = d.dropna(subset=["dist_km", "pop_i", "pop_j"])
    d = d[(d["trips_per_day"] > 0) & (d["dist_km"] > 0) & (d["pop_i"] > 0) & (d["pop_j"] > 0)]
    d["log_trips"] = np.log(d["trips_per_day"])
    d["log_dij"]   = np.log(d["dist_km"])
    d["log_pop_i"] = np.log(d["pop_i"])
    d["log_pop_j"] = np.log(d["pop_j"])
    m = smf.ols("log_trips ~ log_pop_i + log_pop_j + log_dij", data=d).fit()
    return {"n": int(m.nobs), "intercept": m.params["Intercept"],
            "beta_pop_i": m.params["log_pop_i"], "beta_pop_j": m.params["log_pop_j"],
            "gamma": -m.params["log_dij"], "R2": m.rsquared}

def ordered(values, preferred):                                       # keep a sensible band order
    present = [v for v in preferred if v in values]
    return present + sorted(v for v in values if v not in preferred and v != "NA")

AGES    = ordered(set(seg["edad"].unique()),  ["0-25", "25-45", "45-65", "65-100"])
INCOMES = ordered(set(seg["renta"].unique()), ["<10", "10-15", ">15"])

rows = []
for age in AGES:
    for inc in INCOMES:
        sub = seg[(seg["edad"] == age) & (seg["renta"] == inc)]
        if len(sub) == 0:
            continue
        rows.append({"age": age, "income": inc, **fit_cell(sub)})
        print(f"  fitted  age={age:>6}  income={inc:>5}  (n={rows[-1]['n']:,})")

# tidy summary table: one row per age x income cell
grid = pd.DataFrame(rows)
grid.round({"intercept": 3, "beta_pop_i": 3, "beta_pop_j": 3, "gamma": 3, "R2": 4})

  fitted  age=  0-25  income=  <10  (n=225,660)
  fitted  age=  0-25  income=10-15  (n=813,244)
  fitted  age=  0-25  income=  >15  (n=435,328)
  fitted  age= 25-45  income=  <10  (n=297,076)
  fitted  age= 25-45  income=10-15  (n=1,066,749)
  fitted  age= 25-45  income=  >15  (n=553,537)
  fitted  age= 45-65  income=  <10  (n=354,623)
  fitted  age= 45-65  income=10-15  (n=1,296,511)
  fitted  age= 45-65  income=  >15  (n=679,281)
  fitted  age=65-100  income=  <10  (n=180,483)
  fitted  age=65-100  income=10-15  (n=804,317)
  fitted  age=65-100  income=  >15  (n=445,980)


,age,income,n,intercept,beta_pop_i,beta_pop_j,gamma,R2
0,0-25,<10,225660,-2.513,0.217,0.216,0.604,0.2690
1,0-25,10-15,813244,-4.739,0.480,0.493,1.135,0.5732
2,0-25,>15,435328,-5.891,0.459,0.476,0.888,0.4836
3,25-45,<10,297076,-3.384,0.243,0.244,0.607,0.2631
4,25-45,10-15,1066749,-5.355,0.518,0.530,1.210,0.5897
5,25-45,>15,553537,-6.900,0.510,0.526,0.946,0.5007
6,45-65,<10,354623,-3.224,0.235,0.240,0.632,0.2694
7,45-65,10-15,1296511,-4.812,0.504,0.516,1.254,0.5950
8,45-65,>15,679281,-6.285,0.491,0.510,0.999,0.5049
9,65-100,<10,180483,-3.752,0.225,0.228,0.547,0.2366


In [37]:
# headline views: distance-decay gamma and fit R2 as age (rows) x income (cols) grids
gamma_grid = grid.pivot(index="age", columns="income", values="gamma").reindex(index=AGES, columns=INCOMES)
r2_grid    = grid.pivot(index="age", columns="income", values="R2").reindex(index=AGES, columns=INCOMES)

print("model fit  R2  (age x income)")
print(r2_grid.round(4))
print("\ndistance-decay  gamma  (age x income)  --  higher = flows fall off faster with distance")
gamma_grid.round(3)

model fit  R2  (age x income)
income     <10   10-15     >15
age                           
0-25    0.2690  0.5732  0.4836
25-45   0.2631  0.5897  0.5007
45-65   0.2694  0.5950  0.5049
65-100  0.2366  0.5277  0.4427

distance-decay  gamma  (age x income)  --  higher = flows fall off faster with distance


income,<10,10-15,>15
age,,,
0-25,0.604,1.135,0.888
25-45,0.607,1.210,0.946
45-65,0.632,1.254,0.999
65-100,0.547,1.084,0.851


## 8. Same grid, common OD pairs only

The 4×3 grid refit on the OD pairs present in **every** age×income cell — identical n across cells, so coefficients are compared on the same geography.

In [38]:
# valid universe: cross-district pairs with defined, positive population and distance
uni = seg[["id_origin", "id_destination"]].drop_duplicates().astype(str)
uni = uni[uni["id_origin"] != uni["id_destination"]]
uni["pop_i"]   = uni["id_origin"].map(district_pop)
uni["pop_j"]   = uni["id_destination"].map(district_pop)
uni["dist_km"] = haversine_km(uni["id_origin"].map(dist_lon), uni["id_origin"].map(dist_lat),
                              uni["id_destination"].map(dist_lon), uni["id_destination"].map(dist_lat))
uni = uni.dropna(subset=["pop_i", "pop_j", "dist_km"])
uni = uni[(uni["pop_i"] > 0) & (uni["pop_j"] > 0) & (uni["dist_km"] > 0)]
valid = set(zip(uni["id_origin"], uni["id_destination"]))

# common = OD pairs each cell has, intersected across all cells, then kept only if valid
common = None
for age in AGES:
    for inc in INCOMES:
        sub = seg[(seg["edad"] == age) & (seg["renta"] == inc)]
        if len(sub) == 0:
            continue
        pairs = set(zip(sub["id_origin"].astype(str), sub["id_destination"].astype(str)))
        common = pairs if common is None else (common & pairs)
common &= valid
print(f"common OD pairs present in all cells: {len(common):,}")

# refit every cell on exactly these pairs -> identical n, so coefficients compare directly
def fit_common(sub):
    d = sub.copy()
    d["id_origin"] = d["id_origin"].astype(str)
    d["id_destination"] = d["id_destination"].astype(str)
    d["_pair"] = list(zip(d["id_origin"], d["id_destination"]))
    d = d[d["_pair"].isin(common)]
    d["trips_per_day"] = d["n_trips"] / seg_days
    d["dist_km"] = haversine_km(d["id_origin"].map(dist_lon), d["id_origin"].map(dist_lat),
                                d["id_destination"].map(dist_lon), d["id_destination"].map(dist_lat))
    d["pop_i"] = d["id_origin"].map(district_pop)
    d["pop_j"] = d["id_destination"].map(district_pop)
    d["log_trips"] = np.log(d["trips_per_day"])
    d["log_dij"]   = np.log(d["dist_km"])
    d["log_pop_i"] = np.log(d["pop_i"])
    d["log_pop_j"] = np.log(d["pop_j"])
    m = smf.ols("log_trips ~ log_pop_i + log_pop_j + log_dij", data=d).fit()
    return {"n": int(m.nobs), "intercept": m.params["Intercept"],
            "beta_pop_i": m.params["log_pop_i"], "beta_pop_j": m.params["log_pop_j"],
            "gamma": -m.params["log_dij"], "R2": m.rsquared}

rows_c = []
for age in AGES:
    for inc in INCOMES:
        sub = seg[(seg["edad"] == age) & (seg["renta"] == inc)]
        if len(sub) == 0:
            continue
        rows_c.append({"age": age, "income": inc, **fit_common(sub)})

grid_common = pd.DataFrame(rows_c)
grid_common.round({"intercept": 3, "beta_pop_i": 3, "beta_pop_j": 3, "gamma": 3, "R2": 4})

common OD pairs present in all cells: 43,198


,age,income,n,intercept,beta_pop_i,beta_pop_j,gamma,R2
0,0-25,<10,43198,-0.553,0.139,0.124,0.436,0.0559
1,0-25,10-15,43198,-3.807,0.545,0.543,1.279,0.5393
2,0-25,>15,43198,-8.619,0.683,0.677,1.232,0.4818
3,25-45,<10,43198,-1.083,0.167,0.153,0.435,0.0555
4,25-45,10-15,43198,-3.903,0.560,0.554,1.290,0.5480
5,25-45,>15,43198,-9.318,0.727,0.721,1.259,0.4995
6,45-65,<10,43198,0.148,0.108,0.095,0.405,0.0437
7,45-65,10-15,43198,-2.607,0.504,0.500,1.278,0.5363
8,45-65,>15,43198,-8.537,0.701,0.695,1.278,0.4920
9,65-100,<10,43198,-2.078,0.163,0.151,0.437,0.0586


In [39]:
# gamma and R2 across the grid, on the common pairs
gamma_grid_c = grid_common.pivot(index="age", columns="income", values="gamma").reindex(index=AGES, columns=INCOMES)
r2_grid_c    = grid_common.pivot(index="age", columns="income", values="R2").reindex(index=AGES, columns=INCOMES)

print("model fit  R2  (common pairs)")
print(r2_grid_c.round(4))
print("\ndistance-decay  gamma  (common pairs)")
gamma_grid_c.round(3)

model fit  R2  (common pairs)
income     <10   10-15     >15
age                           
0-25    0.0559  0.5393  0.4818
25-45   0.0555  0.5480  0.4995
45-65   0.0437  0.5363  0.4920
65-100  0.0586  0.5065  0.4536

distance-decay  gamma  (common pairs)


income,<10,10-15,>15
age,,,
0-25,0.436,1.279,1.232
25-45,0.435,1.290,1.259
45-65,0.405,1.278,1.278
65-100,0.437,1.307,1.212
